In [ ]:
"""
=============================================================================
 Exercise 3 : Feedforward Neural Network Diagnostic System
              Detecting Learning-Rate, Gradient and Overfitting Failures
              Dataset : Fashion-MNIST (Zalando Research)
=============================================================================
 Models used
   FNN  : single hidden layer feedforward network
   MLP  : 2-3 hidden layer multilayer perceptron
   DFNN : deep feedforward network (6+ hidden layers)

 The system TRAINS the networks and then DIAGNOSES them from real measured
 quantities:
   * epoch-wise training / validation loss and accuracy
   * per-layer gradient magnitudes  (captured with tf.GradientTape)
   * per-layer neuron activation outputs (captured with a probe sub-model)

 Requirements:  tensorflow>=2.10   numpy   matplotlib
 Run:           python nn_failure_diagnosis.py
=============================================================================
"""

import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    PLOTTING = True
except Exception:
    PLOTTING = False

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Results of every experiment that has been run (used by menu option 7)
RESULTS = {}


# =============================================================================
# 1. DATA
# =============================================================================
def load_data(train_subset=12000, test_subset=3000, tiny_train=None):
    """
    Loads Fashion-MNIST, flattens 28x28 -> 784 and scales to [0,1].

    train_subset : sub-sample size used for normal experiments (keeps the
                   menu responsive; set to None to use all 60,000 images)
    tiny_train   : if given, an intentionally small training set used for
                   the OVERFITTING experiment.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.fashion_mnist.load_data()

    x_tr = x_tr.reshape(-1, 784).astype("float32") / 255.0
    x_te = x_te.reshape(-1, 784).astype("float32") / 255.0

    n_train = tiny_train if tiny_train else train_subset
    if n_train:
        x_tr, y_tr = x_tr[:n_train], y_tr[:n_train]
    if test_subset:
        x_te, y_te = x_te[:test_subset], y_te[:test_subset]

    return (x_tr, y_tr), (x_te, y_te)


# =============================================================================
# 2. MODEL BUILDERS  (FNN / MLP / DFNN)
# =============================================================================
def build_model(arch="MLP", activation="relu", init="glorot_uniform",
                dropout=0.0, l2=0.0, width=128):
    """Builds FNN, MLP or DFNN with a chosen activation function."""
    reg = keras.regularizers.l2(l2) if l2 > 0 else None

    if arch == "FNN":
        hidden = [width]
    elif arch == "MLP":
        hidden = [width, width // 2]
    elif arch == "DFNN":
        hidden = [width] * 6            # deep -> exposes vanishing gradients
    else:
        raise ValueError("arch must be FNN, MLP or DFNN")

    model = keras.Sequential(name=f"{arch}_{activation}")
    model.add(layers.Input(shape=(784,)))
    for i, units in enumerate(hidden):
        model.add(layers.Dense(units, activation=activation,
                               kernel_initializer=init,
                               kernel_regularizer=reg,
                               name=f"dense_{i+1}"))
        if dropout > 0:
            model.add(layers.Dropout(dropout, name=f"drop_{i+1}"))
    model.add(layers.Dense(10, activation="softmax", name="output"))
    return model


# =============================================================================
# 3. INSTRUMENTATION
# =============================================================================
LOSS_FN = keras.losses.SparseCategoricalCrossentropy()


class GradientMonitor(keras.callbacks.Callback):
    """Records the mean |gradient| of every Dense kernel after each epoch."""

    def __init__(self, x_probe, y_probe):
        super().__init__()
        self.x = tf.convert_to_tensor(x_probe)
        self.y = tf.convert_to_tensor(y_probe)
        self.history = []          # list of dict{layer_name: mean_abs_grad}

    def on_epoch_end(self, epoch, logs=None):
        with tf.GradientTape() as tape:
            preds = self.model(self.x, training=True)
            loss = LOSS_FN(self.y, preds)
        weights = [w for w in self.model.trainable_weights if "kernel" in w.name]
        grads = tape.gradient(loss, weights)
        rec = {}
        for w, g in zip(weights, grads):
            name = w.name.split("/")[0]
            rec[name] = float(tf.reduce_mean(tf.abs(g))) if g is not None else 0.0
        self.history.append(rec)

    def final(self):
        return self.history[-1] if self.history else {}


def measure_activations(model, x_probe):
    """
    Runs a probe batch through the network and returns, for every hidden
    Dense layer, the fraction of neurons that are dead (output == 0 for
    EVERY sample in the probe batch) plus the mean activation.
    """
    hidden = [l for l in model.layers
              if isinstance(l, layers.Dense) and l.name != "output"]
    if not hidden:
        return {}
    probe = keras.Model(model.inputs, [l.output for l in hidden])
    outs = probe.predict(x_probe, verbose=0)
    if not isinstance(outs, list):
        outs = [outs]

    stats = {}
    for layer, act in zip(hidden, outs):
        active_per_neuron = (act > 1e-8).sum(axis=0)          # per neuron
        dead = float((active_per_neuron == 0).mean())
        stats[layer.name] = {"dead_fraction": dead,
                             "mean_activation": float(act.mean()),
                             "units": act.shape[1]}
    return stats


def train(model, data, lr, epochs, optimizer="sgd", batch_size=128,
          verbose=0, label=""):
    """Compiles + trains a model and returns a bundle of measured metrics."""
    (x_tr, y_tr), (x_te, y_te) = data

    opt = (keras.optimizers.Adam(learning_rate=lr) if optimizer == "adam"
           else keras.optimizers.SGD(learning_rate=lr))
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    probe_x, probe_y = x_tr[:512], y_tr[:512]
    gmon = GradientMonitor(probe_x, probe_y)

    print(f"\n  >> Training {model.name} | optimizer={optimizer} "
          f"| lr={lr} | epochs={epochs}")
    hist = model.fit(x_tr, y_tr,
                     validation_data=(x_te, y_te),
                     epochs=epochs, batch_size=batch_size,
                     callbacks=[gmon], verbose=verbose)

    acts = measure_activations(model, probe_x)

    bundle = {
        "label": label or model.name,
        "model": model.name,
        "lr": lr,
        "optimizer": optimizer,
        "epochs": epochs,
        "train_loss": [float(v) for v in hist.history["loss"]],
        "val_loss": [float(v) for v in hist.history["val_loss"]],
        "train_acc": [float(v) for v in hist.history["accuracy"]],
        "val_acc": [float(v) for v in hist.history["val_accuracy"]],
        "grad_history": gmon.history,
        "grad_final": gmon.final(),
        "activations": acts,
    }
    return bundle


# =============================================================================
# 4. REPORTING HELPERS
# =============================================================================
def print_epoch_table(b):
    print("\n  Epoch |  train_loss |  val_loss |  train_acc |  val_acc")
    print("  ------+-------------+-----------+------------+---------")
    for i in range(len(b["train_loss"])):
        print(f"  {i+1:5d} | {b['train_loss'][i]:11.4f} | "
              f"{b['val_loss'][i]:9.4f} | {b['train_acc'][i]:10.4f} | "
              f"{b['val_acc'][i]:8.4f}")


def print_gradient_table(b):
    g = b["grad_final"]
    if not g:
        return
    print("\n  Final mean |gradient| per layer")
    print("  ---------------------------------")
    for name, val in g.items():
        bar = "#" * min(40, int(-np.log10(val + 1e-30)) * 3) if val > 0 else ""
        print(f"    {name:<10} : {val:.3e}   {bar}")


def print_activation_table(b):
    a = b["activations"]
    if not a:
        return
    print("\n  Neuron activation statistics (probe batch of 512)")
    print("  -------------------------------------------------")
    for name, s in a.items():
        print(f"    {name:<10} : dead {s['dead_fraction']*100:6.2f}% "
              f"of {s['units']:4d} units | mean act {s['mean_activation']:.4f}")


def loss_progress(b):
    """Total drop in training loss across the run."""
    return b["train_loss"][0] - b["train_loss"][-1]


def save_plot(bundles, fname, title):
    if not PLOTTING:
        return
    plt.figure(figsize=(11, 4))
    plt.subplot(1, 2, 1)
    for b in bundles:
        ep = range(1, len(b["train_loss"]) + 1)
        plt.plot(ep, b["train_loss"], label=f"{b['label']} train")
        plt.plot(ep, b["val_loss"], "--", label=f"{b['label']} val")
    plt.title("Loss"); plt.xlabel("epoch"); plt.legend(fontsize=7); plt.grid(alpha=.3)

    plt.subplot(1, 2, 2)
    for b in bundles:
        ep = range(1, len(b["train_acc"]) + 1)
        plt.plot(ep, b["train_acc"], label=f"{b['label']} train")
        plt.plot(ep, b["val_acc"], "--", label=f"{b['label']} val")
    plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend(fontsize=7); plt.grid(alpha=.3)

    plt.suptitle(title); plt.tight_layout()
    plt.savefig(fname, dpi=110); plt.close()
    print(f"\n  [plot saved: {fname}]")


# =============================================================================
# 5. DIAGNOSIS RULES  (decisions are made from the measured numbers)
# =============================================================================
def diagnose_low_lr(b):
    drop = loss_progress(b)
    gain = b["train_acc"][-1] - b["train_acc"][0]
    verdict = ("FAILURE: EXTREMELY LOW LEARNING RATE"
               if drop < 0.05 and gain < 0.05 else "No low-LR failure detected")
    print("\n  DIAGNOSIS")
    print(f"    total training-loss reduction : {drop:.5f}")
    print(f"    total accuracy gain           : {gain:.5f}")
    print(f"    final validation accuracy     : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if drop < 0.05:
        print("    Reason : weight updates w <- w - lr*grad are far too small,")
        print("             so the loss curve is almost flat. Training would")
        print("             need orders of magnitude more epochs to converge.")
        print("    Fix    : raise lr (1e-3 .. 1e-1) or switch to Adam.")
    b["verdict"] = verdict
    return verdict


def diagnose_high_lr(b):
    tl = np.array(b["train_loss"])
    diverged = bool(np.isnan(tl).any() or np.isinf(tl).any() or tl[-1] > tl[0])
    oscillation = float(np.mean(np.abs(np.diff(np.nan_to_num(tl, nan=1e3)))))
    verdict = ("FAILURE: EXTREMELY HIGH LEARNING RATE (divergence/oscillation)"
               if diverged or oscillation > 0.5 else "No high-LR failure detected")
    print("\n  DIAGNOSIS")
    print(f"    NaN / Inf in loss            : {bool(np.isnan(tl).any() or np.isinf(tl).any())}")
    print(f"    first -> last training loss  : {tl[0]:.4f} -> {tl[-1]:.4f}")
    print(f"    mean epoch-to-epoch |change| : {oscillation:.4f}")
    print(f"    final validation accuracy    : {b['val_acc'][-1]:.4f} "
          f"(random guess = 0.1000)")
    print(f"    ==> {verdict}")
    if diverged:
        print("    Reason : the step overshoots the minimum each update, the")
        print("             weights explode and the loss diverges to NaN/Inf.")
        print("    Fix    : reduce lr, use gradient clipping or Adam + warmup.")
    b["verdict"] = verdict
    return verdict


def diagnose_vanishing(b):
    g = b["grad_final"]
    names = list(g.keys())
    vals = np.array([g[n] for n in names])
    first, last = vals[0], vals[-1]
    ratio = last / (first + 1e-30)
    vanishing = bool(first < 1e-4 and ratio > 10)

    verdict = ("FAILURE: VANISHING GRADIENT"
               if vanishing else "No vanishing gradient detected")
    print("\n  DIAGNOSIS")
    print(f"    gradient at FIRST hidden layer : {first:.3e}")
    print(f"    gradient at LAST  hidden layer : {last:.3e}")
    print(f"    last/first ratio               : {ratio:.1f}x")
    print(f"    final validation accuracy      : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if vanishing:
        print("    Reason : sigmoid derivative <= 0.25, so backpropagating")
        print("             through 6 layers shrinks the signal by >= 0.25^6.")
        print("             Early layers receive almost no gradient and stop")
        print("             learning useful features.")
        print("    Fix    : ReLU activations, He initialisation, BatchNorm,")
        print("             residual connections or a shallower network.")
    b["verdict"] = verdict
    return verdict


def diagnose_dead_relu(b):
    acts = b["activations"]
    worst = max(s["dead_fraction"] for s in acts.values()) if acts else 0.0
    avg = float(np.mean([s["dead_fraction"] for s in acts.values()])) if acts else 0.0
    dead = worst > 0.40
    verdict = ("FAILURE: DEAD ReLU NEURONS"
               if dead else "No significant dead-ReLU problem detected")
    print("\n  DIAGNOSIS")
    print(f"    worst layer dead fraction   : {worst*100:.2f}%")
    print(f"    average dead fraction       : {avg*100:.2f}%")
    print(f"    final validation accuracy   : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if dead:
        print("    Reason : a large learning rate drove pre-activations")
        print("             permanently negative; ReLU outputs 0 and its")
        print("             gradient is 0, so those neurons can never recover.")
        print("    Fix    : LeakyReLU / ELU / GELU, smaller lr, He init.")
    b["verdict"] = verdict
    return verdict


def diagnose_overfitting(b):
    gap_acc = b["train_acc"][-1] - b["val_acc"][-1]
    best_val = float(np.min(b["val_loss"]))
    best_ep = int(np.argmin(b["val_loss"])) + 1
    rise = b["val_loss"][-1] - best_val
    over = gap_acc > 0.12 or rise > 0.10
    verdict = "FAILURE: OVERFITTING" if over else "No overfitting detected"
    print("\n  DIAGNOSIS")
    print(f"    final train acc / val acc  : {b['train_acc'][-1]:.4f} / {b['val_acc'][-1]:.4f}")
    print(f"    generalisation gap         : {gap_acc:.4f}")
    print(f"    best val loss              : {best_val:.4f} at epoch {best_ep}")
    print(f"    val-loss rise after best   : {rise:.4f}")
    print(f"    ==> {verdict}")
    if over:
        print("    Reason : the network memorises the small training set;")
        print("             training loss keeps falling while validation loss")
        print(f"             turns upward after epoch {best_ep}.")
        print("    Fix    : dropout, L2 regularisation, early stopping,")
        print("             data augmentation, more training data.")
    b["verdict"] = verdict
    return verdict


def diagnose_stable(b):
    gap = b["train_acc"][-1] - b["val_acc"][-1]
    drop = loss_progress(b)
    g = np.array(list(b["grad_final"].values()))
    dead = max((s["dead_fraction"] for s in b["activations"].values()), default=0)
    healthy = (drop > 0.2 and gap < 0.10 and g.min() > 1e-5 and dead < 0.25
               and b["val_acc"][-1] > 0.80)
    verdict = ("STABLE / HEALTHY CONFIGURATION"
               if healthy else "Configuration is not fully stable")
    print("\n  DIAGNOSIS")
    print(f"    training-loss reduction    : {drop:.4f}")
    print(f"    generalisation gap         : {gap:.4f}")
    print(f"    min layer gradient         : {g.min():.3e}")
    print(f"    worst dead-neuron fraction : {dead*100:.2f}%")
    print(f"    final validation accuracy  : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    b["verdict"] = verdict
    return verdict


# =============================================================================
# 6. EXPERIMENTS  (menu options 1-6)
# =============================================================================
def exp_low_lr(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 1 : EXTREMELY LOW LEARNING RATE   (FNN, SGD, lr = 1e-6)")
    print("=" * 70)
    data = load_data()
    m = build_model("FNN", "relu")
    b = train(m, data, lr=1e-6, epochs=epochs, optimizer="sgd",
              label="Low LR (1e-6)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_low_lr(b)
    save_plot([b], "case1_low_lr.png", "Extremely Low Learning Rate")
    RESULTS["Low LR"] = b
    return b


def exp_high_lr(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 2 : EXTREMELY HIGH LEARNING RATE  (MLP, SGD, lr = 5.0)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu")
    b = train(m, data, lr=5.0, epochs=epochs, optimizer="sgd",
              label="High LR (5.0)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_high_lr(b)
    save_plot([b], "case2_high_lr.png", "Extremely High Learning Rate")
    RESULTS["High LR"] = b
    return b


def exp_vanishing(epochs=10):
    print("\n" + "=" * 70)
    print(" CASE 3 : VANISHING GRADIENT  (DFNN, 6 sigmoid layers, SGD lr=0.01)")
    print("=" * 70)
    data = load_data()
    m = build_model("DFNN", "sigmoid", init="glorot_uniform")
    b = train(m, data, lr=0.01, epochs=epochs, optimizer="sgd",
              label="Vanishing grad (sigmoid DFNN)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_vanishing(b)
    save_plot([b], "case3_vanishing_gradient.png", "Vanishing Gradient")
    RESULTS["Vanishing Gradient"] = b
    return b


def exp_dead_relu(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 4 : DEAD ReLU NEURONS  (MLP, ReLU, SGD lr = 1.0)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu", init="he_normal")
    b = train(m, data, lr=1.0, epochs=epochs, optimizer="sgd",
              label="Dead ReLU (lr=1.0)")
    print_epoch_table(b); print_activation_table(b); print_gradient_table(b)
    diagnose_dead_relu(b)
    save_plot([b], "case4_dead_relu.png", "Dead ReLU Neurons")
    RESULTS["Dead ReLU"] = b
    return b


def exp_overfitting(epochs=40):
    print("\n" + "=" * 70)
    print(" CASE 5 : OVERFITTING  (DFNN, 1000 training images, no regulariser)")
    print("=" * 70)
    data = load_data(tiny_train=1000)
    m = build_model("DFNN", "relu", init="he_normal", width=256)
    b = train(m, data, lr=1e-3, epochs=epochs, optimizer="adam",
              batch_size=32, label="Overfitting")
    print_epoch_table(b)
    diagnose_overfitting(b)
    save_plot([b], "case5_overfitting.png", "Overfitting")
    RESULTS["Overfitting"] = b
    return b


def exp_stable(epochs=12):
    print("\n" + "=" * 70)
    print(" CASE 6 : STABLE CONFIGURATION  (MLP, ReLU+He, Adam 1e-3, dropout")
    print("          0.2 + L2 1e-4)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu", init="he_normal", dropout=0.2, l2=1e-4,
                    width=256)
    b = train(m, data, lr=1e-3, epochs=epochs, optimizer="adam",
              label="Stable")
    print_epoch_table(b); print_gradient_table(b); print_activation_table(b)
    diagnose_stable(b)
    save_plot([b], "case6_stable.png", "Stable Configuration")
    RESULTS["Stable"] = b
    return b


# =============================================================================
# 7. COMPARISON (menu option 7)
# =============================================================================
def compare_all():
    print("\n" + "=" * 70)
    print(" CASE 7 : RUN AND COMPARE ALL FAILURE CASES")
    print("=" * 70)
    exp_low_lr(); exp_high_lr(); exp_vanishing()
    exp_dead_relu(); exp_overfitting(); exp_stable()

    print("\n\n" + "=" * 104)
    print(" COMPARATIVE SUMMARY".center(104))
    print("=" * 104)
    hdr = (f"{'Case':<20}{'Model':<14}{'Opt':<6}{'LR':<9}"
           f"{'TrAcc':<8}{'ValAcc':<8}{'Gap':<8}{'MinGrad':<11}{'Dead%':<8}")
    print(hdr); print("-" * 104)
    for name, b in RESULTS.items():
        g = list(b["grad_final"].values())
        mg = min(g) if g else float("nan")
        dead = max((s["dead_fraction"] for s in b["activations"].values()),
                   default=0.0) * 100
        gap = b["train_acc"][-1] - b["val_acc"][-1]
        print(f"{name:<20}{b['model']:<14}{b['optimizer']:<6}{b['lr']:<9.0e}"
              f"{b['train_acc'][-1]:<8.4f}{b['val_acc'][-1]:<8.4f}{gap:<8.4f}"
              f"{mg:<11.2e}{dead:<8.2f}")
    print("-" * 104)
    print("\n DETECTED BEHAVIOUR / FAILURE")
    print("-" * 104)
    for name, b in RESULTS.items():
        print(f"  {name:<20} -> {b.get('verdict','(not diagnosed)')}")
    print("=" * 104)

    save_plot(list(RESULTS.values()), "case7_comparison.png",
              "All Failure Cases Compared")


# =============================================================================
# 8. MENU
# =============================================================================
MENU = """
==========================================================
      NEURAL NETWORK FAILURE DIAGNOSIS SYSTEM
              (Fashion-MNIST dataset)
==========================================================
 1. Test Extremely Low Learning Rate
 2. Test Extremely High Learning Rate
 3. Detect Vanishing Gradient
 4. Detect Dead ReLU Neurons
 5. Detect Overfitting
 6. Test Stable Neural Network Configuration
 7. Compare All Failure Cases
 8. Exit
==========================================================
"""


def main():
    actions = {"1": exp_low_lr, "2": exp_high_lr, "3": exp_vanishing,
               "4": exp_dead_relu, "5": exp_overfitting, "6": exp_stable,
               "7": compare_all}
    while True:
        print(MENU)
        choice = input("Enter your choice: ").strip()
        if choice == "8":
            print("\nExiting the Neural Network Failure Diagnosis System.")
            break
        fn = actions.get(choice)
        if fn is None:
            print("\n  Invalid choice. Please enter a number from 1 to 8.")
            continue
        try:
            fn()
        except Exception as e:
            print(f"\n  Error while running experiment: {e}")
        input("\nPress Enter to return to the menu...")


if __name__ == "__main__":
    main()


      NEURAL NETWORK FAILURE DIAGNOSIS SYSTEM
              (Fashion-MNIST dataset)
 1. Test Extremely Low Learning Rate
 2. Test Extremely High Learning Rate
 3. Detect Vanishing Gradient
 4. Detect Dead ReLU Neurons
 5. Detect Overfitting
 6. Test Stable Neural Network Configuration
 7. Compare All Failure Cases
 8. Exit

Enter your choice: 1

 CASE 1 : EXTREMELY LOW LEARNING RATE   (FNN, SGD, lr = 1e-6)
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

  >> Training FNN_relu | optimizer=sgd | lr=1e-06 | epochs=8

  Epoch |  train_loss |  val_loss |  train_acc |  val_acc
  ------+-------------+-----------+------------+---------
      1 |      2.5060 |    2.4898 |     0.1277 |   0.1303
      2 |      2.5055 |    2.4893 |     0.1277 |   0.1303
      3 |      2.5050 |    2.4888 |     0.1276 |   0.1303
      4 |      2.5046 |    2.4884 |     0.1276 |

In [ ]:
"""
Exercise 4: ADALINE-Based Adaptive Pattern Classification and Prediction System (Iris dataset)

Classes  : Iris Setosa = 0, Iris Versicolor = 1  (Iris Virginica is discarded)
Features : Sepal Length, Sepal Width, Petal Length, Petal Width
Models   : ADALINE, Single-Layer ADALINE (several linear outputs) and Multi-Input ADALINE,
           all trained with the Widrow-Hoff / LMS (delta) rule on the *linear* output.
           Classification threshold on the linear output = 0.5 (targets are 0 / 1).

Usage    : python exercise4_adaline_iris.py [path_to_iris_csv]
           Looks for Iris.csv / iris.csv / iris.data in the current folder. If none is found,
           scikit-learn's built-in copy of the Iris dataset is used (if scikit-learn is installed).
Needs    : numpy, pandas  (scikit-learn only as an optional fallback for loading data)
"""
import os
import sys
import numpy as np
import pandas as pd

FEATURES = ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]
CLASS_NAME = {0: "Iris-setosa", 1: "Iris-versicolor"}


# ----------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------
def load_iris_binary(path=None):
    cands = ([path] if path else []) + ["Iris.csv", "iris.csv", "iris.data", "iris.data.csv"]
    f = next((p for p in cands if p and os.path.exists(p)), None)
    if f is not None:
        df = pd.read_csv(f, header=None if f.endswith(".data") else 0).dropna(how="all")
        df = df[[c for c in df.columns if str(c).strip().lower() not in ("id", "unnamed: 0")]]
        species = df.iloc[:, -1].astype(str).str.lower()
        keep = species.str.contains("setosa") | species.str.contains("versicolor")
        X = df.iloc[:, :4].apply(pd.to_numeric).values[keep.values]
        y = species[keep].str.contains("versicolor").astype(int).values
        return X.astype(float), y, f
    try:
        from sklearn.datasets import load_iris
        d = load_iris()
        keep = d.target < 2
        return d.data[keep].astype(float), d.target[keep].astype(int), "scikit-learn built-in Iris"
    except ImportError:
        raise SystemExit("Iris data not found. Download Iris.csv from Kaggle/UCI and put it next to this "
                         "script, or run:  python exercise4_adaline_iris.py <csv_path>")


def stratified_split(y, test_frac=0.3, seed=42):
    rng = np.random.default_rng(seed)
    te = []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        te.extend(idx[: int(round(len(idx) * test_frac))])
    te = np.array(sorted(te))
    return np.setdiff1d(np.arange(len(y)), te), te


# ----------------------------------------------------------------------------------------
# ADALINE (single or multiple linear output units = single-layer ADALINE)
# ----------------------------------------------------------------------------------------
class Adaline:
    def __init__(self, n_inputs, n_outputs=1, lr=0.01, epochs=100, seed=0, tol=1e-7):
        rng = np.random.default_rng(seed)
        self.W = rng.uniform(-0.05, 0.05, size=(n_inputs, n_outputs))
        self.b = np.zeros(n_outputs)
        self.lr, self.epochs, self.seed, self.tol = lr, epochs, seed, tol
        self.mse_history = []

    def net(self, X):
        return np.atleast_2d(X) @ self.W + self.b            # linear (identity) activation

    def fit(self, X, Y):
        Y = np.asarray(Y, float).reshape(len(X), -1)
        rng = np.random.default_rng(self.seed + 1)
        self.mse_history = []
        for _ in range(self.epochs):
            for i in rng.permutation(len(X)):
                err = Y[i] - (X[i] @ self.W + self.b)
                self.W += self.lr * np.outer(X[i], err)
                self.b += self.lr * err
            mse = float(np.mean((Y - self.net(X)) ** 2))
            self.mse_history.append(mse)
            if len(self.mse_history) > 1 and abs(self.mse_history[-2] - mse) < self.tol:
                break
        return self

    def predict_class(self, X):
        return (self.net(X)[:, 0] >= 0.5).astype(int)


# ----------------------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------------------
class App:
    pass


app = App()


def setup(path=None):
    X, y, src = load_iris_binary(path)
    tr, te = stratified_split(y)
    app.X, app.y, app.tr, app.te = X, y, tr, te
    app.mu, app.sd = X[tr].mean(0), X[tr].std(0)
    app.sd[app.sd == 0] = 1
    app.Z = (X - app.mu) / app.sd                            # z-score using training statistics
    print(f"Loaded {src}: {len(y)} samples (50 Setosa + 50 Versicolor), 4 features. "
          f"Train {len(tr)}, test {len(te)}.")


def acc(y, p):
    return float((y == p).mean())


def confusion(y, p):
    return (int(((y == 0) & (p == 0)).sum()), int(((y == 0) & (p == 1)).sum()),
            int(((y == 1) & (p == 0)).sum()), int(((y == 1) & (p == 1)).sum()))


def print_conf(y, p):
    tn, fp, fn, tp = confusion(y, p)
    print("Confusion matrix:        Pred Setosa  Pred Versicolor")
    print(f"   Actual Setosa      {tn:>12} {fp:>16}")
    print(f"   Actual Versicolor  {fn:>12} {tp:>16}")


def ask(prompt, default, cast=float):
    s = input(f"{prompt} [default {default}]: ").strip()
    if not s:
        return default
    try:
        return cast(s)
    except ValueError:
        print("   invalid input, using default.")
        return default


def train_clf(cols=(0, 1, 2, 3), lr=0.01, epochs=100, Z=None, y=None, seed=0):
    Z = app.Z[app.tr] if Z is None else Z
    y = app.y[app.tr] if y is None else y
    return Adaline(len(cols), 1, lr, epochs, seed).fit(Z[:, list(cols)], y)


def noisy(Z, sigma, rng):
    return Z + rng.normal(0, sigma, Z.shape)


def get_main():
    if not hasattr(app, "main"):
        app.main = train_clf()
    return app.main


# ----------------------------------------------------------------------------------------
# 1. Binary Pattern Classification
# ----------------------------------------------------------------------------------------
def opt1():
    print("\n=== 1. Binary Pattern Classification (Setosa = 0, Versicolor = 1) ===")
    lr = ask("Learning rate", 0.01)
    ep = ask("Max epochs", 100, int)
    m = train_clf(lr=lr, epochs=ep)
    app.main = m
    Ztr, Zte = app.Z[app.tr], app.Z[app.te]
    print(f"\nEpochs run: {len(m.mse_history)}")
    for e in sorted({1, 5, 10, 25, 50, len(m.mse_history)}):
        if e <= len(m.mse_history):
            print(f"   epoch {e:>3}: MSE = {m.mse_history[e - 1]:.5f}")
    print("\nLearned weights (standardised inputs):")
    for n, w in zip(FEATURES, m.W[:, 0]):
        print(f"   {n:<14}{w:+.4f}")
    print(f"   {'Bias':<14}{m.b[0]:+.4f}")
    ptr, pte = m.predict_class(Ztr), m.predict_class(Zte)
    print(f"\nTraining accuracy: {acc(app.y[app.tr], ptr) * 100:.2f}%   "
          f"Test accuracy: {acc(app.y[app.te], pte) * 100:.2f}%")
    print_conf(app.y[app.te], pte)
    print("\nFirst 8 test samples:")
    print(f"{'idx':>4} {'actual':>16} {'ADALINE output':>15} {'predicted':>16}")
    out = m.net(Zte)[:, 0]
    for i in range(min(8, len(pte))):
        print(f"{i:>4} {CLASS_NAME[app.y[app.te][i]]:>16} {out[i]:>15.4f} {CLASS_NAME[pte[i]]:>16}")
    s = input("\nPredict a new flower? Enter 4 values (SL,SW,PL,PW in cm) or press Enter to skip: ").strip()
    if s:
        try:
            v = np.array([float(t) for t in s.replace(" ", ",").split(",") if t])
            assert len(v) == 4
            o = m.net((v - app.mu) / app.sd)[0, 0]
            print(f"   ADALINE output = {o:.4f} -> {CLASS_NAME[int(o >= 0.5)]}")
        except Exception:
            print("   could not parse four numbers.")


# ----------------------------------------------------------------------------------------
# 2. Multi-Input ADALINE Feature Prediction (regression, single-layer when >1 target)
# ----------------------------------------------------------------------------------------
def opt2():
    print("\n=== 2. Multi-Input ADALINE Feature Prediction ===")
    print("Predict one or more features from the remaining ones (a single-layer ADALINE has one linear output per target).")
    for i, n in enumerate(FEATURES, 1):
        print(f"   {i}. {n}")
    s = input("Target feature number(s), comma separated [default 4]: ").strip() or "4"
    try:
        tg = sorted({int(t) - 1 for t in s.split(",") if t.strip()})
        assert tg and all(0 <= t < 4 for t in tg) and len(tg) < 4
    except Exception:
        print("   invalid selection, using feature 4.")
        tg = [3]
    src = [j for j in range(4) if j not in tg]
    tr, te = app.tr, app.te
    mdl = Adaline(len(src), len(tg), lr=0.01, epochs=300).fit(app.Z[tr][:, src], app.Z[tr][:, tg])
    pred = mdl.net(app.Z[te][:, src])
    print(f"\nInputs : {[FEATURES[j] for j in src]}")
    print(f"Targets: {[FEATURES[j] for j in tg]}   (epochs run: {len(mdl.mse_history)})")
    for k, t in enumerate(tg):
        actual = app.Z[te][:, t] * app.sd[t] + app.mu[t]
        p = pred[:, k] * app.sd[t] + app.mu[t]
        rmse = float(np.sqrt(np.mean((actual - p) ** 2)))
        r2 = 1 - np.sum((actual - p) ** 2) / np.sum((actual - actual.mean()) ** 2)
        print(f"\n--- {FEATURES[t]} ---  RMSE = {rmse:.3f} cm, R^2 = {r2:.3f}")
        print("   weights: " + ", ".join(f"{FEATURES[j]}={mdl.W[a, k]:+.3f}" for a, j in enumerate(src)))
        print(f"   {'idx':>4} {'actual':>8} {'predicted':>10} {'error':>8}")
        for i in range(min(6, len(actual))):
            print(f"   {i:>4} {actual[i]:>8.2f} {p[i]:>10.2f} {p[i] - actual[i]:>+8.2f}")


# ----------------------------------------------------------------------------------------
# 3. Feature Influence Analyzer
# ----------------------------------------------------------------------------------------
def opt3():
    print("\n=== 3. Feature Influence Analyzer ===")
    m = get_main()
    ytr, yte = app.y[app.tr], app.y[app.te]
    w = np.abs(m.W[:, 0])
    base_mse = float(np.mean((yte - m.net(app.Z[app.te])[:, 0]) ** 2))
    print(f"{'feature':<14}{'weight':>9}{'|w| share':>11}{'only-this-feature':>19}{'drop-this-feature':>19}")
    print(f"{'':<14}{'':>9}{'':>11}{'acc / test MSE':>19}{'acc / test MSE':>19}")
    rows = []
    for j, n in enumerate(FEATURES):
        one = train_clf(cols=(j,), epochs=200)
        p1 = one.predict_class(app.Z[app.te][:, [j]])
        mse1 = np.mean((yte - one.net(app.Z[app.te][:, [j]])[:, 0]) ** 2)
        rest = [c for c in range(4) if c != j]
        dr = train_clf(cols=rest, epochs=200)
        p2 = dr.predict_class(app.Z[app.te][:, rest])
        mse2 = np.mean((yte - dr.net(app.Z[app.te][:, rest])[:, 0]) ** 2)
        rows.append((n, m.W[j, 0], w[j] / w.sum() * 100, acc(yte, p1) * 100, mse1, acc(yte, p2) * 100, mse2))
    for n, wt, share, a1, m1, a2, m2 in rows:
        print(f"{n:<14}{wt:>+9.3f}{share:>10.1f}%{a1:>10.1f}% /{m1:>6.3f}{a2:>10.1f}% /{m2:>6.3f}")
    print(f"\n(all four features: test MSE = {base_mse:.3f})")
    best = max(rows, key=lambda r: r[2])
    print(f"Most influential feature by weight: {best[0]}.  A larger rise in MSE when a feature is dropped "
          f"means the network relies on it more.")


# ----------------------------------------------------------------------------------------
# 4. Noise Filtering Challenge
# ----------------------------------------------------------------------------------------
def opt4():
    print("\n=== 4. Noise Filtering Challenge ===")
    print("Gaussian noise (in units of feature standard deviations) is added to the test inputs.")
    k = ask("Number of repeated readings averaged by the filter", 5, int)
    trials = ask("Trials per noise level", 30, int)
    m = get_main()
    yte = app.y[app.te]
    Zte, Ztr, ytr = app.Z[app.te], app.Z[app.tr], app.y[app.tr]
    rng = np.random.default_rng(7)
    print(f"\n{'noise sd':>9}{'A: clean-trained':>18}{'B: + averaging':>16}{'C: noise-trained':>18}   (test accuracy %)")
    for sigma in (0.0, 0.25, 0.5, 1.0, 1.5, 2.0):
        a, b, c = [], [], []
        mC = train_clf(Z=noisy(np.vstack([Ztr] * 5), sigma, rng), y=np.concatenate([ytr] * 5), epochs=60)
        for _ in range(trials):
            a.append(acc(yte, m.predict_class(noisy(Zte, sigma, rng))))
            avg = np.mean([noisy(Zte, sigma, rng) for _ in range(k)], axis=0)    # averaging filter
            b.append(acc(yte, m.predict_class(avg)))
            c.append(acc(yte, mC.predict_class(noisy(Zte, sigma, rng))))
        print(f"{sigma:>9.2f}{np.mean(a) * 100:>18.1f}{np.mean(b) * 100:>16.1f}{np.mean(c) * 100:>18.1f}")
    print("\nA = model trained on clean data, tested on noisy inputs.")
    print(f"B = same model, but each test input is the average of {k} noisy readings (noise sd shrinks by sqrt({k})).")
    print("C = ADALINE trained on noise-corrupted data (LMS learns to average the noise out).")


# ----------------------------------------------------------------------------------------
# 5. Misclassification Pattern Analyzer
# ----------------------------------------------------------------------------------------
def opt5():
    print("\n=== 5. Misclassification Pattern Analyzer ===")
    m = get_main()
    sigma = ask("Noise level for stress test (std units; 0 = clean data only)", 1.0)
    rng = np.random.default_rng(3)
    for label, Z in (("clean data", app.Z), (f"noisy data (sd={sigma})", noisy(app.Z, sigma, rng) if sigma > 0 else None)):
        if Z is None:
            continue
        out = m.net(Z)[:, 0]
        pred = (out >= 0.5).astype(int)
        bad = np.where(pred != app.y)[0]
        print(f"\n--- All 100 samples, {label}: {len(bad)} misclassified ---")
        if len(bad) == 0:
            print("   none - Setosa and Versicolor are linearly separable, so a clean ADALINE gets every sample right.")
            continue
        fp = int(((app.y == 0) & (pred == 1)).sum())
        fn = int(((app.y == 1) & (pred == 0)).sum())
        print(f"   Setosa -> predicted Versicolor: {fp}   Versicolor -> predicted Setosa: {fn}")
        print(f"   {'idx':>4} {'actual':>16} {'output':>8}  SL/SW/PL/PW (cm, original sample)")
        for i in bad[:12]:
            r = app.X[i]
            print(f"   {i:>4} {CLASS_NAME[app.y[i]]:>16} {out[i]:>8.3f}  {r[0]:.1f}/{r[1]:.1f}/{r[2]:.1f}/{r[3]:.1f}")
        if len(bad) > 12:
            print(f"   ... {len(bad) - 12} more")
        good = np.setdiff1d(np.arange(len(app.y)), bad)
        print("   Mean z-score, misclassified vs correct (which features push samples across the boundary):")
        for j, n in enumerate(FEATURES):
            print(f"      {n:<14}{Z[bad, j].mean():>+7.2f} vs {Z[good, j].mean():>+7.2f}")
        print(f"   Mean |output - 0.5|: misclassified {np.mean(np.abs(out[bad] - 0.5)):.3f}, correct {np.mean(np.abs(out[good] - 0.5)):.3f}")


# ----------------------------------------------------------------------------------------
# 6. Decision Boundary Analyzer
# ----------------------------------------------------------------------------------------
def opt6():
    print("\n=== 6. Decision Boundary Analyzer ===")
    m = get_main()
    w, b = m.W[:, 0], m.b[0]
    w_o = w / app.sd
    b_o = b - np.sum(w * app.mu / app.sd)
    print("Boundary of the 4-feature model, in original units (cm):")
    print("   " + " ".join(f"{c:+.4f}*{n.replace(' ', '')}" for c, n in zip(w_o, FEATURES)) + f" {b_o:+.4f} = 0.5")
    dist = (m.net(app.Z)[:, 0] - 0.5) / np.linalg.norm(w)       # signed distance in standardised space
    side_ok = ((dist >= 0) == (app.y == 1))
    print(f"Samples on the correct side: {int(side_ok.sum())} / {len(app.y)}")
    print(f"Margin (distance to boundary, std units): Setosa mean {np.abs(dist[app.y == 0]).mean():.2f}, "
          f"Versicolor mean {np.abs(dist[app.y == 1]).mean():.2f}")
    print("\n5 samples closest to the boundary:")
    print(f"   {'idx':>4} {'actual':>16} {'distance':>9}")
    for i in np.argsort(np.abs(dist))[:5]:
        print(f"   {i:>4} {CLASS_NAME[app.y[i]]:>16} {dist[i]:>+9.3f}")

    # 2-D boundary on Petal Length (x) vs Petal Width (y)
    cols = (2, 3)
    m2 = train_clf(cols=cols, epochs=200)
    w2, b2 = m2.W[:, 0] / app.sd[list(cols)], m2.b[0] - np.sum(m2.W[:, 0] * app.mu[list(cols)] / app.sd[list(cols)])
    print("\n2-feature ADALINE (Petal Length, Petal Width):")
    print(f"   boundary: {w2[0]:+.4f}*PL {w2[1]:+.4f}*PW {b2:+.4f} = 0.5")
    if abs(w2[1]) > 1e-9:
        print(f"   i.e. PW = {-w2[0] / w2[1]:+.4f}*PL {(0.5 - b2) / w2[1]:+.4f}")
    xs, ys = app.X[:, cols[0]], app.X[:, cols[1]]
    W_, H_ = 60, 18
    x0, x1, y0, y1 = xs.min() - 0.2, xs.max() + 0.2, ys.min() - 0.1, ys.max() + 0.1
    grid = [[" "] * W_ for _ in range(H_)]
    if abs(w2[1]) > 1e-9:
        for c in range(W_):
            xv = x0 + (x1 - x0) * c / (W_ - 1)
            yv = (0.5 - b2 - w2[0] * xv) / w2[1]
            r = int(round((y1 - yv) / (y1 - y0) * (H_ - 1)))
            if 0 <= r < H_:
                grid[r][c] = "."
    for xv, yv, cl in zip(xs, ys, app.y):
        c = int(round((xv - x0) / (x1 - x0) * (W_ - 1)))
        r = int(round((y1 - yv) / (y1 - y0) * (H_ - 1)))
        grid[r][c] = "S" if cl == 0 else "V"
    print(f"\n   Petal Width ^   (S = Setosa, V = Versicolor, . = decision boundary)")
    for row in grid:
        print("   |" + "".join(row))
    print("   +" + "-" * W_ + "> Petal Length")


# ----------------------------------------------------------------------------------------
# 7. Comparative Analysis
# ----------------------------------------------------------------------------------------
def opt7():
    print("\n=== 7. Comparative Analysis ===")
    Ztr, Zte = app.Z[app.tr], app.Z[app.te]
    ytr, yte = app.y[app.tr], app.y[app.te]
    rows = []

    def add(name, model, cols, one_hot=False):
        Xte = Zte[:, list(cols)]
        if one_hot:
            p = np.argmax(model.net(Xte), axis=1)
            mse = np.mean((np.eye(2)[yte] - model.net(Xte)) ** 2)
        else:
            p = model.predict_class(Xte)
            mse = np.mean((yte - model.net(Xte)[:, 0]) ** 2)
        rows.append((name, acc(yte, p) * 100, float(mse), len(model.mse_history)))

    add("ADALINE, 1 input (Petal Length)", train_clf((2,), epochs=200), (2,))
    add("ADALINE, 2 inputs (PL, PW)", train_clf((2, 3), epochs=200), (2, 3))
    add("Multi-input ADALINE (4 inputs)", train_clf((0, 1, 2, 3), epochs=200), (0, 1, 2, 3))
    sl = Adaline(4, 2, 0.01, 200).fit(Ztr, np.eye(2)[ytr])
    add("Single-layer ADALINE (4 in, 2 out)", sl, (0, 1, 2, 3), one_hot=True)
    add("Multi-input, high lr (0.1)", train_clf((0, 1, 2, 3), lr=0.1, epochs=200), (0, 1, 2, 3))
    add("Multi-input, low lr (0.001)", train_clf((0, 1, 2, 3), lr=0.001, epochs=200), (0, 1, 2, 3))
    print(f"\n{'model':<38}{'test acc %':>11}{'test MSE':>10}{'epochs':>8}")
    for n, a, mse, ep in rows:
        print(f"{n:<38}{a:>11.1f}{mse:>10.4f}{ep:>8}")
    print("\nAccuracy is near-perfect for every model because Setosa/Versicolor are linearly separable; MSE and "
          "epochs-to-converge are what separate the models. More informative inputs lower MSE, and the learning "
          "rate trades convergence speed against stability.")


MENU = """
============ ADALINE Iris Classification System ============
 1. Binary Pattern Classification
 2. Multi-Input ADALINE Feature Prediction
 3. Feature Influence Analyzer
 4. Noise Filtering Challenge
 5. Misclassification Pattern Analyzer
 6. Decision Boundary Analyzer
 7. Comparative Analysis
 8. Exit
============================================================"""
ACTIONS = {"1": opt1, "2": opt2, "3": opt3, "4": opt4, "5": opt5, "6": opt6, "7": opt7}


def main():
    setup(sys.argv[1] if len(sys.argv) > 1 else None)
    while True:
        print(MENU)
        try:
            c = input("Enter your choice (1-8): ").strip()
        except EOFError:
            break
        if c == "8":
            print("Goodbye!")
            break
        fn = ACTIONS.get(c)
        if fn is None:
            print("Invalid choice, please enter a number from 1 to 8.")
            continue
        fn()


if __name__ == "__main__":
    main()

Loaded scikit-learn built-in Iris: 100 samples (50 Setosa + 50 Versicolor), 4 features. Train 70, test 30.

============ ADALINE Iris Classification System ============
 1. Binary Pattern Classification
 2. Multi-Input ADALINE Feature Prediction
 3. Feature Influence Analyzer
 4. Noise Filtering Challenge
 5. Misclassification Pattern Analyzer
 6. Decision Boundary Analyzer
 7. Comparative Analysis
 8. Exit
Enter your choice (1-8): 1

=== 1. Binary Pattern Classification (Setosa = 0, Versicolor = 1) ===
Learning rate [default 0.01]: 2
Max epochs [default 100]: 3

Epochs run: 3
   epoch   1: MSE = 10634724814771124525110140340168906240978956003091380376321697532503222459099001126912.00000
   epoch   3: MSE = 74961898938862108014083115220270219906845915661111690000527980136478385751784879310407589906707985534388174058952874548737545131625931786940657130423874595261669998055330606429421175860964275653974353789883020100107093744428096755962301523785940992.00000

Learned weights (standardis